# Qwen Gradient Difference

This notebook implements **standard Gradient Difference (GD) unlearning only** for the trained Qwen3.5-2B classifier. It starts each scenario from the same frozen Original model state, updates only the trained LoRA adapter and binary classification head, and reports the predefined epoch-5 model.

Completed Full Retraining outputs are loaded only as a behavioural reference during final evaluation. They are never modified and never used to select a GD checkpoint.


## 1. Gradient Difference Method

### 1.1 Purpose

The experiment asks whether a small task-specific adaptation can be changed so that its behaviour on a deletion request approaches Full Retraining while retained predictive utility is preserved. The pretrained Qwen base stays frozen; this is adaptation-level unlearning of the trained LoRA and classification head.

Only three prespecified deletion requests are run: Recipient Withdrawal (426 training rows), Invalid Consent (4,148), and Retention Expiry (6,262). These represent small, medium, and large requests.


### 1.2 Objective

For trainable parameters $\theta$, standard Gradient Difference minimises

$$
L_{GD}(\theta)=L_{retain}(\theta)-\lambda L_{forget}(\theta),
\qquad \lambda=1.
$$

- $\theta$: the trainable LoRA and binary classification-head parameters;
- $L_{retain}$: ordinary classification cross-entropy on sampled retained-training examples;
- $L_{forget}$: ordinary classification cross-entropy on the requested training forget examples;
- $\lambda$: the fixed forget-loss weight.

Minimising the retained term encourages retained predictive behaviour. Subtracting the forget term means gradient descent increases classification loss on forgotten examples.


### 1.3 How This Differs from Gradient Ascent

Gradient Ascent uses only the deletion request,

$$L_{GA}=-L_{forget},$$

whereas GD also includes a retained-data anchor. It is also distinct from ordinary joint fitting,

$$L_{joint}=L_{retain}+L_{forget},$$

which improves performance on both sets rather than unlearning the forget set. No Retain-Set Fine-Tuning, Gradient Ascent, Full Retraining, NPO, or safety-constrained training code appears in this notebook.


## 2. Environment Verification

The cells below verify the exact successful A100 environment. They deliberately do not install or upgrade packages. A mismatch stops execution before model loading or training.


### 2.1 GPU and CUDA


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version as package_version

import gc
import hashlib
import json
import os
import random
import shutil
import time

import numpy as np
import pandas as pd
import torch

from IPython.display import Markdown, display

# Training is intentionally restricted to the hardware used by the
# completed reference experiment, making runtime comparisons interpretable.
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is unavailable. Use the final A100-SXM4-80GB RunPod; "
        "this notebook will not repair or replace the environment."
    )

DEVICE = torch.device("cuda")
GPU_NAME = torch.cuda.get_device_name(0)

gpu_checks = {
    "GPU is NVIDIA A100-SXM4-80GB": "A100-SXM4-80GB" in GPU_NAME,
    "PyTorch is 2.8.0+cu128": torch.__version__.startswith("2.8.0+cu128"),
    "Torch CUDA runtime is 12.8": torch.version.cuda == "12.8",
}

display(pd.DataFrame(
    {"Check": gpu_checks.keys(), "Passed": gpu_checks.values()}
).assign(Status=lambda frame: frame["Passed"].map({True: "PASS", False: "FAIL"})))

if not all(gpu_checks.values()):
    raise RuntimeError(
        f"Wrong GPU/CUDA environment: GPU={GPU_NAME!r}, "
        f"torch={torch.__version__!r}, CUDA={torch.version.cuda!r}. "
        "Stop here and use the successful final RunPod image."
    )


### 2.2 Package Versions


In [ ]:
# Unsloth must be imported before Transformers so its Qwen patches are active.
import unsloth
from unsloth import FastVisionModel

from transformers import AutoTokenizer
from peft import PeftModel
from scipy.stats import ks_2samp
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
)
from torch import nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

EXPECTED_PACKAGES = {
    "unsloth": "2026.8.22",
    "transformers": "5.2.0",
    "triton": "3.4.0",
}

package_checks = {
    package: package_version(package) == expected
    for package, expected in EXPECTED_PACKAGES.items()
}

display(pd.DataFrame([
    {
        "Package": package,
        "Expected": expected,
        "Observed": package_version(package),
        "Status": "PASS" if package_checks[package] else "FAIL",
    }
    for package, expected in EXPECTED_PACKAGES.items()
]))

if not all(package_checks.values()):
    raise RuntimeError(
        "Package versions do not match the completed Full Retraining stack. "
        "Stop here; do not upgrade packages inside this notebook."
    )


### 2.3 Reproducibility


In [ ]:
SEED = 42

def reset_seed(seed=SEED):
    # Resetting all generators makes independently restored scenarios repeatable.
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

reset_seed()
print("Global seed:", SEED)


## 3. Frozen Original Qwen

### 3.1 Locate Frozen Artefacts

The canonical final-submission bundle is authoritative. The search supports both the repository checkout and its RunPod mount without changing filenames or substituting an older run.


In [ ]:
def locate_final_submission():
    candidates = [
        Path.cwd().resolve(),
        *Path.cwd().resolve().parents,
        Path("/workspace/qub-machine-unlearning"),
    ]

    for candidate in candidates:
        possible_roots = [
            candidate if candidate.name == "final_submission" else None,
            candidate / "final_submission",
            candidate / "code" / "final_submission",
        ]
        for root in possible_roots:
            if root is None:
                continue
            marker = root / "data" / "final" / "kidney_transplant_assessments.csv"
            if marker.is_file():
                return root.resolve()

    raise FileNotFoundError("Could not locate code/final_submission.")

FINAL = locate_final_submission()
MODEL_ROOT = FINAL / "models" / "qwen"
RESULT_ROOT = FINAL / "results" / "qwen"

BASELINE_ADAPTER = MODEL_ROOT / "baseline" / "adapter"
BASELINE_HEAD = MODEL_ROOT / "baseline" / "binary_classification_head.pt"
BASELINE_RESULTS = RESULT_ROOT / "original"
FULL_RESULTS = RESULT_ROOT / "full_retraining"

DATA_PATH = FINAL / "data" / "final" / "kidney_transplant_assessments.csv"
FEATURE_PATH = FINAL / "data" / "final" / "classifier_feature_list.json"
SPLIT_PATH = FINAL / "processed_data" / "split_assignments.csv"
MEMBERSHIP_PATH = FINAL / "processed_data" / "deletion_scenario_membership.csv"

METHOD_MODEL_ROOT = MODEL_ROOT / "gradient_difference"
METHOD_RESULT_ROOT = RESULT_ROOT / "unlearning" / "gradient_difference"


In [ ]:
required_inputs = {
    "assessment data": DATA_PATH,
    "feature contract": FEATURE_PATH,
    "permanent split": SPLIT_PATH,
    "deletion memberships": MEMBERSHIP_PATH,
    "adapter configuration": BASELINE_ADAPTER / "adapter_config.json",
    "classification head": BASELINE_HEAD,
    "baseline configuration": BASELINE_RESULTS / "experiment_configuration.json",
    "frozen threshold": BASELINE_RESULTS / "selected_threshold.json",
    "serialisation specification": BASELINE_RESULTS / "serialisation_specification.json",
    "saved baseline probabilities": BASELINE_RESULTS / "retained_test_probabilities.csv",
}

adapter_weights = [
    path for path in [
        BASELINE_ADAPTER / "adapter_model.safetensors",
        BASELINE_ADAPTER / "adapter_model.bin",
    ]
    if path.is_file()
]

missing_inputs = [path for path in required_inputs.values() if not path.is_file()]
if missing_inputs or len(adapter_weights) != 1:
    raise FileNotFoundError(
        "Frozen Original Qwen bundle is incomplete. Missing files:\n  "
        + "\n  ".join(map(str, missing_inputs))
        + f"\nAdapter weight candidates found: {adapter_weights}"
    )

BASELINE_ADAPTER_WEIGHTS = adapter_weights[0]
display(pd.DataFrame([
    {"Artefact": name, "Path": str(path), "Exists": path.is_file()}
    for name, path in required_inputs.items()
]))


### 3.2 Baseline Configuration


In [ ]:
baseline_config = json.loads(
    (BASELINE_RESULTS / "experiment_configuration.json").read_text(encoding="utf-8")
)
baseline_threshold = json.loads(
    (BASELINE_RESULTS / "selected_threshold.json").read_text(encoding="utf-8")
)
baseline_serialisation = json.loads(
    (BASELINE_RESULTS / "serialisation_specification.json").read_text(encoding="utf-8")
)
adapter_config = json.loads(
    (BASELINE_ADAPTER / "adapter_config.json").read_text(encoding="utf-8")
)

MODEL_ID = baseline_config["model_id"]
MAX_SEQ_LENGTH = int(baseline_config["max_seq_length"])
FROZEN_THRESHOLD = float(baseline_threshold["threshold"])

assert baseline_config["run_id"] == "20260829T151430Z"
assert MODEL_ID == "unsloth/Qwen3.5-2B-Base"
assert MAX_SEQ_LENGTH == 216
assert FROZEN_THRESHOLD == 0.55
assert int(adapter_config["r"]) == 16
assert int(adapter_config["lora_alpha"]) == 16
assert float(adapter_config["lora_dropout"]) == 0.0
assert adapter_config["base_model_name_or_path"] == MODEL_ID

display(pd.Series({
    "Baseline run": baseline_config["run_id"],
    "Model": MODEL_ID,
    "Maximum sequence length": MAX_SEQ_LENGTH,
    "Frozen threshold": FROZEN_THRESHOLD,
    "LoRA rank": adapter_config["r"],
    "LoRA alpha": adapter_config["lora_alpha"],
    "LoRA dropout": adapter_config["lora_dropout"],
}, name="Frozen value").to_frame())


### 3.3 Load Original Qwen

The function below follows the successful Full Retraining environment: 16-bit `FastVisionModel`, no 4-bit quantisation, and no gradient checkpointing. The two-class output layer is installed **before** loading the trained PEFT adapter, then the saved head is restored. This cell defines reconstruction; the model is instantiated once after the cached inputs are ready.


In [ ]:
class FP32ClassificationHead(nn.Linear):
    # Keeping the small classifier in FP32 avoids avoidable loss instability.
    def forward(self, hidden_states):
        return F.linear(hidden_states.float(), self.weight, self.bias)

def clear_device_cache():
    gc.collect()
    torch.cuda.empty_cache()

def load_original_qwen(tokenizer):
    reset_seed()
    clear_device_cache()

    base_model, _ = FastVisionModel.from_pretrained(
        MODEL_ID,
        load_in_4bit=False,
        load_in_16bit=True,
        max_seq_length=MAX_SEQ_LENGTH,
        use_gradient_checkpointing=False,
    )

    # The base represents pretrained knowledge and is outside this
    # adaptation-level unlearning experiment.
    for parameter in base_model.parameters():
        parameter.requires_grad = False

    old_head = base_model.get_output_embeddings()
    base_model.set_output_embeddings(FP32ClassificationHead(
        old_head.in_features,
        2,
        bias=False,
        device=old_head.weight.device,
        dtype=torch.float32,
    ))
    base_model.config.num_labels = 2
    base_model.config.pad_token_id = tokenizer.pad_token_id

    # is_trainable=True preserves the already trained adapter as the
    # starting point; it does not create a fresh random LoRA adapter.
    model = PeftModel.from_pretrained(
        base_model,
        BASELINE_ADAPTER,
        is_trainable=True,
    )

    head_state = torch.load(BASELINE_HEAD, map_location="cpu", weights_only=True)
    current_state = model.state_dict()
    bad_head_keys = [
        name for name, value in head_state.items()
        if name not in current_state or current_state[name].shape != value.shape
    ]
    if not head_state or bad_head_keys:
        raise RuntimeError(f"Binary-head restore failed: {bad_head_keys[:5]}")
    model.load_state_dict(head_state, strict=False)

    # Explicit permissions make the scientific scope auditable even if a
    # future PEFT release changes its default requires_grad behaviour.
    for name, parameter in model.named_parameters():
        parameter.requires_grad = ("lora_" in name or "lm_head" in name)
        if parameter.requires_grad:
            parameter.data = parameter.data.float()

    model.config.use_cache = False
    return model.to(DEVICE)


### 3.4 Verify Reconstructed Baseline


In [ ]:
def capture_trainable(model):
    # Only this compact state is copied between scenarios; the 2.2B base is not.
    return {
        name: parameter.detach().cpu().clone()
        for name, parameter in model.named_parameters()
        if parameter.requires_grad
    }

def restore_trainable(model, state):
    named_parameters = dict(model.named_parameters())
    with torch.no_grad():
        for name, value in state.items():
            if name not in named_parameters:
                raise KeyError(f"Missing trainable parameter: {name}")
            named_parameters[name].copy_(value.to(named_parameters[name].device))

def state_fingerprint(state):
    digest = hashlib.sha256()
    for name, value in sorted(state.items()):
        tensor = value.detach().cpu().contiguous()
        digest.update(name.encode("utf-8"))
        digest.update(str(tensor.dtype).encode("ascii"))
        digest.update(str(tuple(tensor.shape)).encode("ascii"))
        digest.update(tensor.numpy().tobytes())
    return digest.hexdigest()


In [ ]:
def assert_trainable_scope(model):
    trainable_names = [
        name for name, parameter in model.named_parameters()
        if parameter.requires_grad
    ]
    forbidden = [
        name for name in trainable_names
        if "lora_" not in name and "lm_head" not in name
    ]

    assert trainable_names, "No trainable parameters were found."
    assert any("lora_" in name for name in trainable_names)
    assert any("lm_head" in name for name in trainable_names)
    assert not forbidden, f"Frozen Qwen base parameters became trainable: {forbidden[:5]}"
    return trainable_names


The actual 64-row reproduction check is run in Section 5.5, after the permanent test rows have been serialised and cached. It compares IDs, labels, and probabilities and performs no optimisation.


## 4. Dataset and Deletion Scenarios

### 4.1 Load the Frozen Dataset


In [ ]:
assessments = pd.read_csv(DATA_PATH)
feature_contract = json.loads(FEATURE_PATH.read_text(encoding="utf-8"))

TARGET = feature_contract["target"]
FEATURES = feature_contract["classifier_features"]

EXPECTED_FEATURES = [
    "recipient_age", "donor_age", "donor_type", "kidney_failure_cause",
    "previous_transplant", "dialysis_months", "abo_compatibility_category",
    "hla_mismatch_count", "antibody_risk_score", "cold_ischaemia_hours",
    "days_since_transplant", "creatinine_mg_dl", "creatinine_change_pct",
    "urine_output_ml_24h", "tacrolimus_level_ng_ml",
    "medication_adherence_pct", "infection_indicator", "previous_rejection",
]

assert len(assessments) == 60_000
assert assessments["assessment_id"].is_unique
assert TARGET == "acute_rejection_within_30_days"
assert FEATURES == EXPECTED_FEATURES
print("Frozen assessment rows:", len(assessments))


### 4.2 Recreate the Permanent Split


In [ ]:
split_assignments = pd.read_csv(SPLIT_PATH)

data = assessments.merge(
    split_assignments[["recipient_id", "donor_id", "split"]],
    on=["recipient_id", "donor_id"],
    how="left",
    validate="many_to_one",
)
assert data["split"].notna().all()

split_frames = {
    split_name: data.loc[data["split"].eq(split_name)].copy().reset_index(drop=True)
    for split_name in ["train", "validation", "test"]
}
assert {name: len(frame) for name, frame in split_frames.items()} == {
    "train": 42_024,
    "validation": 8_988,
    "test": 8_988,
}

for frame in split_frames.values():
    frame["assessment_id"] = frame["assessment_id"].astype(str)
    frame["label"] = frame[TARGET].astype("int64")

display(pd.DataFrame([
    {"Split": name, "Rows": len(frame), "Positive prevalence": frame["label"].mean()}
    for name, frame in split_frames.items()
]))


### 4.3 Load Exact Deletion Membership


In [ ]:
# This file is frozen experimental input. No deletion rule is re-evaluated here.
membership = pd.read_csv(MEMBERSHIP_PATH)
membership["assessment_id"] = membership["assessment_id"].astype(str)

assert not membership.duplicated(["scenario", "assessment_id"]).any()

SCENARIO_LABELS = {
    "recipient_withdrawal": "Recipient Withdrawal",
    "donor_withdrawal": "Donor Withdrawal",
    "invalid_consent": "Invalid Consent",
    "hospital_removal": "Hospital Removal",
    "retention_expiry": "Retention Expiry",
}

# The original full order is retained solely for compatible sampling seeds.
ALL_SCENARIOS = [
    "recipient_withdrawal",
    "donor_withdrawal",
    "invalid_consent",
    "hospital_removal",
    "retention_expiry",
]
assert set(membership["scenario"]) == set(ALL_SCENARIOS)


### 4.4 Final Three Scenarios


In [ ]:
# These names are explicit so an accidental slice cannot change the experiment.
FINAL_SCENARIOS = [
    "recipient_withdrawal",
    "invalid_consent",
    "retention_expiry",
]

EXPECTED_FORGET = {
    "recipient_withdrawal": 426,
    "invalid_consent": 4_148,
    "retention_expiry": 6_262,
}

scenario_sets = {}
for scenario in FINAL_SCENARIOS:
    scenario_membership = membership.loc[membership["scenario"].eq(scenario)]
    membership_ids = {
        kind: set(scenario_membership.loc[
            scenario_membership["membership_type"].eq(kind), "assessment_id"
        ])
        for kind in ["training_forget", "deleted_validation", "deleted_test"]
    }

    training_forget = split_frames["train"].loc[
        split_frames["train"]["assessment_id"].isin(membership_ids["training_forget"])
    ].copy()
    retained_train = split_frames["train"].loc[
        ~split_frames["train"]["assessment_id"].isin(membership_ids["training_forget"])
    ].copy()
    retained_validation = split_frames["validation"].loc[
        ~split_frames["validation"]["assessment_id"].isin(membership_ids["deleted_validation"])
    ].copy()
    retained_test = split_frames["test"].loc[
        ~split_frames["test"]["assessment_id"].isin(membership_ids["deleted_test"])
    ].copy()

    scenario_sets[scenario] = {
        "training_forget": training_forget,
        "retained_train": retained_train,
        "retained_validation": retained_validation,
        "retained_test": retained_test,
        "deleted_validation_ids": membership_ids["deleted_validation"],
        "deleted_test_ids": membership_ids["deleted_test"],
    }


In [ ]:
scenario_audit = []
for scenario, parts in scenario_sets.items():
    forget_ids = set(parts["training_forget"]["assessment_id"])
    retain_ids = set(parts["retained_train"]["assessment_id"])

    assert len(forget_ids) == EXPECTED_FORGET[scenario]
    assert len(forget_ids) + len(retain_ids) == 42_024
    assert forget_ids.isdisjoint(retain_ids)

    scenario_audit.append({
        "Scenario": SCENARIO_LABELS[scenario],
        "Forget rows": len(forget_ids),
        "Retained training rows": len(retain_ids),
        "Total training rows": len(forget_ids) + len(retain_ids),
        "Disjoint": forget_ids.isdisjoint(retain_ids),
    })

display(pd.DataFrame(scenario_audit))


## 5. Qwen Input Preparation

### 5.1 Frozen Feature Order


In [ ]:
FEATURE_LABELS = baseline_serialisation["display_labels"]
BINARY_FEATURES = {
    "previous_transplant",
    "infection_indicator",
    "previous_rejection",
}

assert baseline_serialisation["feature_order"] == FEATURES
assert baseline_serialisation["target_included"] is False
assert baseline_serialisation["identifiers_included"] is False

display(pd.DataFrame({
    "Position": range(1, len(FEATURES) + 1),
    "Feature": FEATURES,
    "Display label": [FEATURE_LABELS[feature] for feature in FEATURES],
}))


### 5.2 Deterministic Text Serialisation


In [ ]:
def format_feature_value(feature, value):
    if pd.isna(value):
        return "missing"
    if feature in BINARY_FEATURES:
        return "yes" if int(value) == 1 else "no"
    if isinstance(value, (float, np.floating)):
        return f"{float(value):.4f}".rstrip("0").rstrip(".")
    return str(value).strip()

def serialize_assessment(row):
    # Feature order is the saved model contract, not dataframe column order.
    return "\n".join(
        f"{FEATURE_LABELS[feature]}: {format_feature_value(feature, row[feature])}."
        for feature in FEATURES
    )


In [ ]:
# Serialise the complete dataset once so every scenario sees identical text.
serialised = data.copy()
serialised["assessment_id"] = serialised["assessment_id"].astype(str)
serialised["text"] = serialised.apply(serialize_assessment, axis=1)
text_by_id = serialised.set_index("assessment_id")["text"]

for parts in scenario_sets.values():
    for key in ["training_forget", "retained_train", "retained_validation", "retained_test"]:
        frame = parts[key]
        frame["text"] = frame["assessment_id"].map(text_by_id)
        assert frame["text"].notna().all()

print(serialised.loc[0, "text"])


### 5.3 Tokenise All Assessments Once


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    BASELINE_ADAPTER,
    local_files_only=True,
)
tokenizer.padding_side = "right"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

TOKEN_CACHE = {}
TOKENISE_BATCH_SIZE = 2_048
all_ids = serialised["assessment_id"].tolist()
all_texts = serialised["text"].tolist()

for start in tqdm(range(0, len(all_texts), TOKENISE_BATCH_SIZE), desc="Tokenising once"):
    end = min(start + TOKENISE_BATCH_SIZE, len(all_texts))
    encoded = tokenizer(
        all_texts[start:end],
        add_special_tokens=True,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding=False,
    )
    for assessment_id, token_ids in zip(all_ids[start:end], encoded["input_ids"]):
        # int32 halves cache storage relative to int64; collators cast only a batch.
        TOKEN_CACHE[assessment_id] = torch.tensor(token_ids, dtype=torch.int32)


In [ ]:
token_lengths = np.fromiter(
    (len(token_ids) for token_ids in TOKEN_CACHE.values()),
    dtype=np.int32,
    count=len(TOKEN_CACHE),
)

token_audit = pd.Series({
    "Cached rows": len(TOKEN_CACHE),
    "Maximum token length": int(token_lengths.max()),
    "Mean token length": float(token_lengths.mean()),
    "Frozen maximum sequence length": MAX_SEQ_LENGTH,
}, name="Value")
display(token_audit.to_frame())

assert len(TOKEN_CACHE) == 60_000
assert set(TOKEN_CACHE) == set(all_ids)
assert int(token_lengths.max()) <= MAX_SEQ_LENGTH


### 5.4 Cached DataLoader


In [ ]:
class CachedDataset(Dataset):
    def __init__(self, frame):
        frame = frame.reset_index(drop=True)
        self.ids = frame["assessment_id"].astype(str).tolist()
        self.labels = frame["label"].astype(int).tolist()

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, index):
        assessment_id = self.ids[index]
        return {
            "assessment_id": assessment_id,
            "input_ids": TOKEN_CACHE[assessment_id],
            "label": self.labels[index],
        }

def cached_collate(rows):
    # Dynamic padding avoids paying for 216 tokens when a batch is shorter.
    sequences = [row["input_ids"].long() for row in rows]
    input_ids = pad_sequence(
        sequences,
        batch_first=True,
        padding_value=tokenizer.pad_token_id,
    )
    return {
        "input_ids": input_ids,
        "attention_mask": (input_ids != tokenizer.pad_token_id).long(),
        "labels": torch.tensor([row["label"] for row in rows], dtype=torch.long),
        "assessment_id": [row["assessment_id"] for row in rows],
    }


In [ ]:
NUM_WORKERS = 2

def make_loader(frame, batch_size, shuffle=False, generator=None):
    return DataLoader(
        CachedDataset(frame),
        batch_size=batch_size,
        shuffle=shuffle,
        generator=generator,
        collate_fn=cached_collate,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=(NUM_WORKERS > 0),
    )

def final_token_logits(model, batch):
    input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
    attention_mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
    sequence_logits = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
    ).logits
    final_indices = attention_mask.sum(dim=1) - 1
    logits = sequence_logits[
        torch.arange(input_ids.shape[0], device=DEVICE),
        final_indices,
    ]
    if logits.shape != (input_ids.shape[0], 2):
        raise RuntimeError(f"Unexpected logits shape: {tuple(logits.shape)}")
    return logits


### 5.5 Load Once and Run the Baseline Diagnostic


In [ ]:
# This is the notebook's only 2.2B base-model load.
MODEL = load_original_qwen(tokenizer)
TRAINABLE_NAMES = assert_trainable_scope(MODEL)

FROZEN_BASELINE_STATE = capture_trainable(MODEL)
FROZEN_BASELINE_SHA256 = state_fingerprint(FROZEN_BASELINE_STATE)

display(pd.Series({
    "Total parameters": sum(p.numel() for p in MODEL.parameters()),
    "Trainable parameters": sum(p.numel() for p in MODEL.parameters() if p.requires_grad),
    "Trainable tensors": len(TRAINABLE_NAMES),
    "Baseline SHA-256": FROZEN_BASELINE_SHA256,
}, name="Value").to_frame())


In [ ]:
@torch.inference_mode()
def predict_fixed(model, frame, batch_size, description):
    model.eval()
    rows = []
    for batch in tqdm(make_loader(frame, batch_size), desc=description, leave=False):
        probabilities = torch.softmax(final_token_logits(model, batch).float(), dim=1)[:, 1]
        rows.extend({
            "assessment_id": assessment_id,
            "label": int(label),
            "probability_class_1": float(probability),
        } for assessment_id, label, probability in zip(
            batch["assessment_id"],
            batch["labels"].tolist(),
            probabilities.cpu().tolist(),
        ))
    return pd.DataFrame(rows)

def verify_reconstructed_baseline(sample_size=64):
    saved = pd.read_csv(BASELINE_RESULTS / "retained_test_probabilities.csv")
    saved["assessment_id"] = saved["assessment_id"].astype(str)
    expected = saved.head(sample_size).copy()
    expected_ids = expected["assessment_id"].tolist()

    indexed_test = split_frames["test"].set_index("assessment_id")
    if not set(expected_ids).issubset(indexed_test.index):
        raise RuntimeError("Saved baseline IDs are not in the permanent test split.")
    probe = indexed_test.loc[expected_ids].reset_index()
    observed = predict_fixed(MODEL, probe, 64, "Verify Original Qwen")

    differences = np.abs(
        observed["probability_class_1"].to_numpy()
        - expected["probability_class_1"].to_numpy()
    )
    report = {
        "Compared rows": len(observed),
        "IDs align": observed["assessment_id"].tolist() == expected_ids,
        "Labels align": np.array_equal(observed["label"], expected["label"]),
        "Maximum absolute probability difference": float(differences.max()),
        "Mean absolute probability difference": float(differences.mean()),
    }
    report["Passed"] = (
        report["IDs align"]
        and report["Labels align"]
        and np.allclose(
            observed["probability_class_1"],
            expected["probability_class_1"],
            atol=1e-5,
            rtol=1e-4,
        )
    )
    display(pd.Series(report, name="Baseline diagnostic").to_frame())
    if not report["Passed"]:
        raise RuntimeError(
            "Reconstructed Original Qwen materially disagrees with saved prediction evidence. "
            "Do not start unlearning."
        )
    return report

BASELINE_VERIFICATION = verify_reconstructed_baseline()


## 6. Gradient Difference Setup

### 6.1 Forget and Retain Data


One GD epoch is one complete pass over the scenario's training forget request. Every forget row is paired with exactly one row sampled from that scenario's `retained_train`; validation, test, deleted-validation, and deleted-test rows are ineligible.


### 6.2 Deterministic Retained Sampling


In [ ]:
def retained_sampling_seed(scenario, epoch):
    scenario_index = ALL_SCENARIOS.index(scenario)
    return SEED + 10_000 * (scenario_index + 1) + epoch

def paired_retain_indices(scenario, epoch):
    parts = scenario_sets[scenario]
    sample_seed = retained_sampling_seed(scenario, epoch)
    rng = np.random.default_rng(sample_seed)
    indices = rng.integers(
        0,
        len(parts["retained_train"]),
        size=len(parts["training_forget"]),
    )

    sampled_ids = set(parts["retained_train"].iloc[indices]["assessment_id"])
    forget_ids = set(parts["training_forget"]["assessment_id"])
    assert sampled_ids.issubset(set(parts["retained_train"]["assessment_id"]))
    assert sampled_ids.isdisjoint(forget_ids)
    assert sampled_ids.isdisjoint(parts["deleted_validation_ids"])
    assert sampled_ids.isdisjoint(parts["deleted_test_ids"])
    assert len(indices) == len(parts["training_forget"])
    return indices


### 6.3 Verify Sampling


In [ ]:
sampling_audit_rows = []
for scenario in FINAL_SCENARIOS:
    for epoch in range(1, 6):
        indices = paired_retain_indices(scenario, epoch)
        forget_examples = len(scenario_sets[scenario]["training_forget"])
        sampled_examples = len(indices)
        assert forget_examples == sampled_examples
        sampling_audit_rows.append({
            "Scenario": SCENARIO_LABELS[scenario],
            "Epoch": epoch,
            "Forget examples": forget_examples,
            "Sampled retain examples": sampled_examples,
            "Unique retain examples": len(np.unique(indices)),
            "Sampling seed": retained_sampling_seed(scenario, epoch),
        })

sampling_audit = pd.DataFrame(sampling_audit_rows)
display(sampling_audit)


### 6.4 Gradient Difference Configuration


In [ ]:
GD_CONFIG = {
    "method": "standard_gradient_difference",
    "optimizer": "AdamW",
    "learning_rate": 1e-5,
    "weight_decay": 0.0,
    "lambda": 1.0,
    "physical_pair_batch": 32,
    "gradient_accumulation": 1,
    "effective_pair_batch": 32,
    "epochs": 5,
    "gradient_clip_norm": 1.0,
    "model_selection": "none",
    "primary_model": "epoch 5",
    "baseline_run_id": "20260829T151430Z",
}

assert GD_CONFIG["lambda"] == 1.0
assert GD_CONFIG["physical_pair_batch"] * GD_CONFIG["gradient_accumulation"] == 32
assert GD_CONFIG["epochs"] == 5

GD_CONFIG_SHA256 = hashlib.sha256(
    json.dumps(GD_CONFIG, sort_keys=True).encode("utf-8")
).hexdigest()

display(pd.Series(GD_CONFIG, name="Frozen GD value").to_frame())
print("Configuration SHA-256:", GD_CONFIG_SHA256)


The trajectory is fixed at exactly five epochs. There is no early stopping and no validation-, test-, KS-, or Full-Retraining-based model selection. Epoch 5 is the primary model by prior design; full validation after epochs 1–4 is therefore unnecessary.


## 7. Gradient Difference Implementation

### 7.1 Paired Dataset


In [ ]:
class PairedCachedDataset(Dataset):
    def __init__(self, forget_frame, retain_frame, retain_indices):
        self.forget = forget_frame.reset_index(drop=True)
        self.retain = retain_frame.reset_index(drop=True)
        self.retain_indices = np.asarray(retain_indices, dtype=np.int64)
        assert len(self.forget) == len(self.retain_indices)

    def __len__(self):
        return len(self.forget)

    def __getitem__(self, index):
        forget_row = self.forget.iloc[index]
        retain_row = self.retain.iloc[int(self.retain_indices[index])]
        return {
            "forget_id": str(forget_row["assessment_id"]),
            "forget_label": int(forget_row["label"]),
            "retain_id": str(retain_row["assessment_id"]),
            "retain_label": int(retain_row["label"]),
        }


In [ ]:
def paired_cached_collate(rows):
    forget_ids = [row["forget_id"] for row in rows]
    retain_ids = [row["retain_id"] for row in rows]

    # Forget examples are first, so one Qwen output can be split reliably.
    combined_ids = forget_ids + retain_ids
    sequences = [TOKEN_CACHE[assessment_id].long() for assessment_id in combined_ids]
    input_ids = pad_sequence(
        sequences,
        batch_first=True,
        padding_value=tokenizer.pad_token_id,
    )
    return {
        "input_ids": input_ids,
        "attention_mask": (input_ids != tokenizer.pad_token_id).long(),
        "forget_labels": torch.tensor(
            [row["forget_label"] for row in rows], dtype=torch.long
        ),
        "retain_labels": torch.tensor(
            [row["retain_label"] for row in rows], dtype=torch.long
        ),
        "pair_count": len(rows),
    }


In [ ]:
def make_paired_loader(scenario, epoch):
    parts = scenario_sets[scenario]
    sample_seed = retained_sampling_seed(scenario, epoch)
    paired_dataset = PairedCachedDataset(
        parts["training_forget"],
        parts["retained_train"],
        paired_retain_indices(scenario, epoch),
    )
    return DataLoader(
        paired_dataset,
        batch_size=GD_CONFIG["physical_pair_batch"],
        shuffle=True,
        generator=torch.Generator().manual_seed(sample_seed + 1),
        collate_fn=paired_cached_collate,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=(NUM_WORKERS > 0),
    )


### 7.2 Combined Qwen Forward Pass

For a physical pair batch of 32, the collator supplies 32 forget plus 32 retain sequences. One Qwen invocation processes all 64; splitting the logits does not alter the GD objective and avoids a second expensive forward pass.


In [ ]:
def paired_logits(model, batch):
    input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
    attention_mask = batch["attention_mask"].to(DEVICE, non_blocking=True)

    # This is the single Qwen invocation for the complete pair batch.
    sequence_logits = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
    ).logits
    final_indices = attention_mask.sum(dim=1) - 1
    logits = sequence_logits[
        torch.arange(input_ids.shape[0], device=DEVICE),
        final_indices,
    ]

    pair_count = int(batch["pair_count"])
    if logits.shape != (2 * pair_count, 2):
        raise RuntimeError(f"Unexpected paired logits shape: {tuple(logits.shape)}")
    return logits[:pair_count], logits[pair_count:]


### 7.3 Forget Loss

`forget_loss` is ordinary unweighted two-class cross-entropy over the complete forget request, one pass per epoch.


In [ ]:
def classification_cross_entropy(logits, labels):
    return F.cross_entropy(logits.float(), labels)


### 7.4 Retain Loss

`retain_loss` uses the same cross-entropy definition on the one-to-one sampled retained partners.


### 7.5 Gradient Difference Objective

The two losses remain explicit in both the smoke test and training loop. With the frozen $\lambda=1$, the implemented line is exactly `objective = retain_loss - forget_loss`.


### 7.6 Verify Objective


In [ ]:
def require_finite(value, name, scenario, epoch, step):
    if not bool(torch.isfinite(value).all().item()):
        raise RuntimeError(
            f"Non-finite {name}: scenario={scenario}, epoch={epoch}, step={step}."
        )


### 7.7 Smoke Test


In [ ]:
def run_gd_smoke_test():
    scenario = "recipient_withdrawal"
    restore_trainable(MODEL, FROZEN_BASELINE_STATE)
    assert state_fingerprint(capture_trainable(MODEL)) == FROZEN_BASELINE_SHA256
    assert_trainable_scope(MODEL)

    parameters = [p for p in MODEL.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(
        parameters,
        lr=GD_CONFIG["learning_rate"],
        weight_decay=GD_CONFIG["weight_decay"],
    )
    batch = next(iter(make_paired_loader(scenario, epoch=1)))
    forget_labels = batch["forget_labels"].to(DEVICE, non_blocking=True)
    retain_labels = batch["retain_labels"].to(DEVICE, non_blocking=True)
    forget_logits, retain_logits = paired_logits(MODEL, batch)

    forget_loss = classification_cross_entropy(forget_logits, forget_labels)
    retain_loss = classification_cross_entropy(retain_logits, retain_labels)

    # This visible subtraction is the scientific method being tested.
    objective = retain_loss - forget_loss

    for name, value in {
        "forget loss": forget_loss,
        "retain loss": retain_loss,
        "objective": objective,
    }.items():
        require_finite(value, name, scenario, 1, 1)

    objective.backward()
    grad_norm = torch.nn.utils.clip_grad_norm_(
        parameters, GD_CONFIG["gradient_clip_norm"]
    )
    require_finite(grad_norm, "gradient norm", scenario, 1, 1)
    optimizer.step()

    report = {
        "Forget loss": float(forget_loss.item()),
        "Retain loss": float(retain_loss.item()),
        "Objective": float(objective.item()),
        "Gradient norm": float(grad_norm.item()),
    }
    assert np.isclose(
        report["Objective"],
        report["Retain loss"] - report["Forget loss"],
        atol=1e-7,
        rtol=0,
    )

    # The disposable update must have no effect on a final scenario.
    restore_trainable(MODEL, FROZEN_BASELINE_STATE)
    MODEL.zero_grad(set_to_none=True)
    assert state_fingerprint(capture_trainable(MODEL)) == FROZEN_BASELINE_SHA256
    del optimizer
    clear_device_cache()
    display(pd.Series(report, name="Disposable GD smoke test").to_frame())
    print("Smoke test passed; frozen baseline restored exactly.")

run_gd_smoke_test()


### 7.8 Safe Output and Resume Paths


In [ ]:
def scenario_paths(scenario):
    model_dir = METHOD_MODEL_ROOT / scenario
    result_dir = METHOD_RESULT_ROOT / scenario
    return {
        "model_dir": model_dir,
        "adapter": model_dir / "adapter",
        "head": model_dir / "binary_classification_head.pt",
        "result_dir": result_dir,
        "checkpoint": result_dir / "_resume_checkpoint.pt",
        "complete": result_dir / "COMPLETE.json",
    }

def valid_complete_payload(payload, scenario):
    return (
        payload.get("status") == "complete"
        and payload.get("scenario") == scenario
        and payload.get("method") == GD_CONFIG["method"]
        and payload.get("baseline_sha256") == FROZEN_BASELINE_SHA256
        and payload.get("configuration_sha256") == GD_CONFIG_SHA256
        and int(payload.get("selected_epoch", -1)) == 5
    )


In [ ]:
def archive_partial(paths, scenario, reason):
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    archive = METHOD_RESULT_ROOT / "_partial_archive" / stamp / scenario
    archive.mkdir(parents=True, exist_ok=False)
    for source in [paths["model_dir"], paths["result_dir"]]:
        if source.exists():
            shutil.move(str(source), str(archive / source.name))
    print(f"Archived incomplete/stale run ({reason}): {archive}")

def prepare_scenario(scenario):
    paths = scenario_paths(scenario)

    if paths["complete"].is_file():
        payload = json.loads(paths["complete"].read_text(encoding="utf-8"))
        if valid_complete_payload(payload, scenario):
            return "skip"
        archive_partial(paths, scenario, "invalid COMPLETE marker")
    elif paths["checkpoint"].is_file():
        checkpoint = torch.load(paths["checkpoint"], map_location="cpu", weights_only=False)
        valid_checkpoint = (
            checkpoint.get("scenario") == scenario
            and checkpoint.get("method") == GD_CONFIG["method"]
            and checkpoint.get("baseline_sha256") == FROZEN_BASELINE_SHA256
            and checkpoint.get("configuration_sha256") == GD_CONFIG_SHA256
            # An epoch-5 checkpoint can resume directly into final evaluation
            # if RunPod stopped after optimisation but before COMPLETE was written.
            and 1 <= int(checkpoint.get("epoch", -1)) <= GD_CONFIG["epochs"]
        )
        if valid_checkpoint:
            return "resume"
        archive_partial(paths, scenario, "incompatible resume checkpoint")
    elif paths["model_dir"].exists() or paths["result_dir"].exists():
        archive_partial(paths, scenario, "no valid completion/checkpoint marker")

    paths["model_dir"].mkdir(parents=True, exist_ok=True)
    paths["result_dir"].mkdir(parents=True, exist_ok=True)
    return "run"


### 7.9 Lightweight Epoch Checkpoints


In [ ]:
def save_resume_checkpoint(path, scenario, epoch, model, optimizer, history, elapsed_seconds):
    payload = {
        "method": GD_CONFIG["method"],
        "scenario": scenario,
        "epoch": epoch,
        "current_trainable_state": capture_trainable(model),
        "optimizer_state": optimizer.state_dict(),
        "history": history,
        "elapsed_unlearning_seconds": elapsed_seconds,
        "baseline_sha256": FROZEN_BASELINE_SHA256,
        "configuration": GD_CONFIG,
        "configuration_sha256": GD_CONFIG_SHA256,
    }
    temporary = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, temporary)
    temporary.replace(path)

def optimizer_to_device(optimizer):
    for state in optimizer.state.values():
        for key, value in state.items():
            if torch.is_tensor(value):
                state[key] = value.to(DEVICE)


### 7.10 Fixed Five-Epoch Training Loop


In [ ]:
def initialise_gd_training(model, scenario, resume):
    paths = scenario_paths(scenario)
    restore_trainable(model, FROZEN_BASELINE_STATE)
    assert state_fingerprint(capture_trainable(model)) == FROZEN_BASELINE_SHA256
    assert_trainable_scope(model)

    parameters = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(
        parameters,
        lr=GD_CONFIG["learning_rate"],
        weight_decay=GD_CONFIG["weight_decay"],
    )
    start_epoch = 1
    history = []
    elapsed_seconds = 0.0

    if resume:
        checkpoint = torch.load(
            paths["checkpoint"], map_location="cpu", weights_only=False
        )
        checkpoint_matches = (
            checkpoint["baseline_sha256"] == FROZEN_BASELINE_SHA256
            and checkpoint["configuration_sha256"] == GD_CONFIG_SHA256
            and checkpoint["scenario"] == scenario
        )
        if not checkpoint_matches:
            raise RuntimeError("Resume checkpoint does not match this experiment.")

        restore_trainable(model, checkpoint["current_trainable_state"])
        optimizer.load_state_dict(checkpoint["optimizer_state"])
        optimizer_to_device(optimizer)
        start_epoch = int(checkpoint["epoch"]) + 1
        history = checkpoint["history"]
        elapsed_seconds = float(checkpoint["elapsed_unlearning_seconds"])
        print("Resuming at epoch", start_epoch)

    return paths, parameters, optimizer, start_epoch, history, elapsed_seconds


In [ ]:
def train_one_gd_epoch(model, scenario, epoch, optimizer, parameters):
    loader = make_paired_loader(scenario, epoch)
    model.train()
    optimizer.zero_grad(set_to_none=True)
    forget_total = 0.0
    retain_total = 0.0
    objective_total = 0.0
    rows_seen = 0

    # Timing encloses optimisation only. Tokenisation, model loading,
    # final evaluation, and checkpoint file writing are excluded.
    torch.cuda.synchronize()
    epoch_started = time.perf_counter()

    for step, batch in enumerate(tqdm(loader, desc=f"GD {scenario} epoch {epoch}"), 1):
        forget_labels = batch["forget_labels"].to(DEVICE, non_blocking=True)
        retain_labels = batch["retain_labels"].to(DEVICE, non_blocking=True)
        forget_logits, retain_logits = paired_logits(model, batch)

        forget_loss = classification_cross_entropy(forget_logits, forget_labels)
        retain_loss = classification_cross_entropy(retain_logits, retain_labels)

        # Standard Gradient Difference with the frozen lambda of one.
        objective = retain_loss - forget_loss

        require_finite(forget_loss, "forget loss", scenario, epoch, step)
        require_finite(retain_loss, "retain loss", scenario, epoch, step)
        require_finite(objective, "objective", scenario, epoch, step)

        objective.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(
            parameters, GD_CONFIG["gradient_clip_norm"]
        )
        require_finite(grad_norm, "gradient norm", scenario, epoch, step)
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)

        pair_count = len(forget_labels)
        forget_total += float(forget_loss.detach().item()) * pair_count
        retain_total += float(retain_loss.detach().item()) * pair_count
        objective_total += float(objective.detach().item()) * pair_count
        rows_seen += pair_count

    torch.cuda.synchronize()
    epoch_seconds = time.perf_counter() - epoch_started
    if rows_seen != len(scenario_sets[scenario]["training_forget"]):
        raise RuntimeError("An epoch did not cover the complete forget request.")

    row = {
        "scenario": scenario,
        "epoch": epoch,
        "mean_forget_ce": forget_total / rows_seen,
        "mean_retain_ce": retain_total / rows_seen,
        "mean_gd_objective": objective_total / rows_seen,
        "forget_examples": rows_seen,
        "sampled_retain_examples": rows_seen,
        "sampling_seed": retained_sampling_seed(scenario, epoch),
    }
    return row, epoch_seconds


In [ ]:
def train_gd(model, scenario, resume=False):
    (
        paths,
        parameters,
        optimizer,
        start_epoch,
        history,
        elapsed_seconds,
    ) = initialise_gd_training(model, scenario, resume)

    torch.cuda.reset_peak_memory_stats()
    for epoch in range(start_epoch, GD_CONFIG["epochs"] + 1):
        row, epoch_seconds = train_one_gd_epoch(
            model, scenario, epoch, optimizer, parameters
        )
        elapsed_seconds += epoch_seconds
        history.append(row)
        display(pd.DataFrame([row]).round(6))

        # Save only the compact trainable state after each complete epoch.
        save_resume_checkpoint(
            paths["checkpoint"],
            scenario,
            epoch,
            model,
            optimizer,
            history,
            elapsed_seconds,
        )

    if not history or int(history[-1]["epoch"]) != 5:
        raise RuntimeError("Standard GD did not reach its required epoch-5 endpoint.")

    summary = {
        **GD_CONFIG,
        "scenario": scenario,
        "selected_epoch": 5,
        "epochs_executed": 5,
        "training_seconds": float(elapsed_seconds),
        "peak_gpu_memory_gib": float(torch.cuda.max_memory_allocated() / 1024**3),
        "gpu": GPU_NAME,
        "baseline_sha256": FROZEN_BASELINE_SHA256,
        "configuration_sha256": GD_CONFIG_SHA256,
        "selection_rule": "fixed epoch 5; evaluation evidence is not used for selection",
        "objective": "retain CE - forget CE",
    }
    return history, summary


## 8. Evaluation

Evaluation occurs once, after the fixed epoch-5 model has been trained. It supplies evidence only and cannot alter the selected epoch.

### 8.1 Retained-Test Utility

PR-AUC, Balanced Accuracy, BCE, F1, AUROC, Precision, Recall, and Specificity use the Original model's frozen threshold of 0.55.


In [ ]:
EVALUATION_BATCHES = [256, 128, 64, 32]

@torch.inference_mode()
def evaluate_at_batch(model, frame, description, batch_size):
    model.eval()
    rows = []
    for batch in tqdm(make_loader(frame, batch_size), desc=description, leave=False):
        probabilities = torch.softmax(final_token_logits(model, batch).float(), dim=1)[:, 1]
        rows.extend({
            "assessment_id": assessment_id,
            "label": int(label),
            "probability_class_1": float(probability),
        } for assessment_id, label, probability in zip(
            batch["assessment_id"],
            batch["labels"].tolist(),
            probabilities.cpu().tolist(),
        ))
    return pd.DataFrame(rows)

def evaluate_frame(model, frame, description):
    for batch_size in EVALUATION_BATCHES:
        try:
            return evaluate_at_batch(model, frame, description, batch_size), batch_size
        except torch.cuda.OutOfMemoryError:
            clear_device_cache()
            print(f"Evaluation OOM at {batch_size}; retrying with a smaller batch.")
    raise RuntimeError("Evaluation did not fit even at batch size 32.")


In [ ]:
def utility_metrics(predictions):
    labels = predictions["label"].to_numpy()
    probabilities = predictions["probability_class_1"].to_numpy()
    predicted = (probabilities >= FROZEN_THRESHOLD).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, predicted, labels=[0, 1]).ravel()
    return {
        "n": len(labels),
        "pr_auc": average_precision_score(labels, probabilities),
        "balanced_accuracy": balanced_accuracy_score(labels, predicted),
        "binary_cross_entropy": log_loss(labels, probabilities, labels=[0, 1]),
        "f1": f1_score(labels, predicted, zero_division=0),
        "auroc": roc_auc_score(labels, probabilities),
        "precision": precision_score(labels, predicted, zero_division=0),
        "recall": recall_score(labels, predicted, zero_division=0),
        "specificity": tn / (tn + fp),
        "threshold": FROZEN_THRESHOLD,
    }


### 8.2 Forget-Set Probabilities

The complete training forget request is evaluated, with exactly the same probability schema used by the Original and Full Retraining experiments.


### 8.3 Truth Ratio

For the true label probability $p_{true}$, Truth Ratio is

$$rac{p_{incorrect}+10^{-12}}{p_{true}+10^{-12}}.$$

The epsilon prevents division by zero and matches the established project definition.


In [ ]:
def truth_components(labels, probabilities, epsilon=1e-12):
    p_true = np.where(labels == 1, probabilities, 1 - probabilities)
    p_incorrect = 1 - p_true
    truth_ratio = (p_incorrect + epsilon) / (p_true + epsilon)
    return p_true, p_incorrect, truth_ratio


### 8.4 KS Comparison with Full Retraining

The two-sample KS statistic compares GD and matching Full Retraining Truth Ratio distributions on identical forget IDs. Lower KS indicates closer observed distributional behaviour. Its p-value is supporting evidence: a high p-value alone does not prove erasure, parameter-level deletion, or legal compliance.


In [ ]:
def load_full_retraining_reference(scenario):
    result_dir = FULL_RESULTS / scenario
    complete_path = result_dir / "COMPLETE.json"
    forget_path = result_dir / "forget_set_probabilities.csv"
    if not complete_path.is_file() or not forget_path.is_file():
        raise FileNotFoundError(f"Missing completed Full Retraining reference: {result_dir}")

    complete = json.loads(complete_path.read_text(encoding="utf-8"))
    if complete.get("status") != "complete":
        raise RuntimeError(f"Full Retraining is not complete: {scenario}")
    forget = pd.read_csv(forget_path)
    forget["assessment_id"] = forget["assessment_id"].astype(str)
    return complete, forget


In [ ]:
def forgetting_evidence(approximate, full_retraining):
    approximate = approximate.copy()
    full_retraining = full_retraining.copy()
    approximate["assessment_id"] = approximate["assessment_id"].astype(str)
    full_retraining["assessment_id"] = full_retraining["assessment_id"].astype(str)

    assert set(approximate["assessment_id"]) == set(full_retraining["assessment_id"])
    joined = approximate.merge(
        full_retraining,
        on="assessment_id",
        suffixes=("_gd", "_full"),
        validate="one_to_one",
    )
    assert len(joined) == len(approximate) == len(full_retraining)
    assert np.array_equal(joined["label_gd"], joined["label_full"])

    labels = joined["label_gd"].to_numpy()
    gd_probability = joined["probability_class_1_gd"].to_numpy()
    full_probability = joined["probability_class_1_full"].to_numpy()
    gd_true, gd_incorrect, gd_ratio = truth_components(labels, gd_probability)
    full_true, full_incorrect, full_ratio = truth_components(labels, full_probability)
    ks_result = ks_2samp(gd_ratio, full_ratio, alternative="two-sided", method="auto")

    values = pd.DataFrame({
        "assessment_id": joined["assessment_id"],
        "label": labels,
        "gd_probability": gd_probability,
        "full_retraining_probability": full_probability,
        "gd_p_true": gd_true,
        "gd_p_incorrect": gd_incorrect,
        "gd_truth_ratio": gd_ratio,
        "full_retraining_p_true": full_true,
        "full_retraining_p_incorrect": full_incorrect,
        "full_retraining_truth_ratio": full_ratio,
        "epsilon": 1e-12,
    })
    metrics = {
        "forget_rows": len(values),
        "ks_statistic": float(ks_result.statistic),
        "ks_p_value": float(ks_result.pvalue),
    }
    return values, metrics


### 8.5 Computational Runtime

Primary GD time begins immediately before optimisation batches and stops immediately after them. Model loading, global tokenisation, final evaluation, and saving are excluded. Formal speed-up is Full Retraining training time divided by GD optimisation time; GPU hardware is recorded because timing is hardware dependent.


### 8.6 Save the Epoch-5 Model and Evidence


In [ ]:
def save_selected_model(model, scenario):
    paths = scenario_paths(scenario)
    paths["adapter"].mkdir(parents=True, exist_ok=True)
    model.save_pretrained(paths["adapter"], safe_serialization=True)
    tokenizer.save_pretrained(paths["adapter"])

    head_state = {
        name: value.detach().cpu()
        for name, value in model.state_dict().items()
        if "lm_head" in name
    }
    if not head_state:
        raise RuntimeError("Binary classification head was not found while saving.")
    torch.save(head_state, paths["head"])


In [ ]:
def save_evaluation_tables(paths, history, metrics, retained_predictions,
                           forget_predictions, truth_values, forgetting):
    pd.DataFrame(history).to_csv(
        paths["result_dir"] / "training_history.csv", index=False
    )
    pd.DataFrame([metrics]).to_csv(
        paths["result_dir"] / "retained_test_metrics.csv", index=False
    )
    retained_predictions.to_csv(
        paths["result_dir"] / "retained_test_probabilities.csv", index=False
    )
    forget_predictions.to_csv(
        paths["result_dir"] / "forget_set_probabilities.csv", index=False
    )
    truth_values.to_csv(
        paths["result_dir"] / "truth_ratio_values.csv", index=False
    )
    pd.DataFrame([forgetting]).to_csv(
        paths["result_dir"] / "forgetting_metrics.csv", index=False
    )


In [ ]:
def evaluate_scenario_evidence(model, scenario, summary):
    parts = scenario_sets[scenario]
    retained_predictions, retained_batch = evaluate_frame(
        model, parts["retained_test"], f"GD retained test: {scenario}"
    )
    forget_predictions, forget_batch = evaluate_frame(
        model, parts["training_forget"], f"GD forget set: {scenario}"
    )
    metrics = utility_metrics(retained_predictions)
    full_complete, full_forget = load_full_retraining_reference(scenario)
    truth_values, forgetting = forgetting_evidence(forget_predictions, full_forget)

    full_seconds = float(full_complete["training_seconds"])
    runtime = {
        "runtime_boundary": "optimisation batches only",
        "gradient_difference_training_seconds": summary["training_seconds"],
        "full_retraining_training_seconds": full_seconds,
        "speed_up": full_seconds / summary["training_seconds"],
        "gpu": GPU_NAME,
        "retained_test_evaluation_batch": retained_batch,
        "forget_evaluation_batch": forget_batch,
    }
    return {
        "retained_predictions": retained_predictions,
        "forget_predictions": forget_predictions,
        "metrics": metrics,
        "truth_values": truth_values,
        "forgetting": forgetting,
        "runtime": runtime,
    }


In [ ]:
def publish_scenario_result(model, scenario, history, summary, evidence):
    paths = scenario_paths(scenario)
    save_selected_model(model, scenario)
    save_evaluation_tables(
        paths,
        history,
        evidence["metrics"],
        evidence["retained_predictions"],
        evidence["forget_predictions"],
        evidence["truth_values"],
        evidence["forgetting"],
    )
    (paths["result_dir"] / "runtime.json").write_text(
        json.dumps(evidence["runtime"], indent=2), encoding="utf-8"
    )
    (paths["result_dir"] / "configuration.json").write_text(
        json.dumps(summary, indent=2), encoding="utf-8"
    )

    complete = {
        "status": "complete",
        "method": GD_CONFIG["method"],
        "scenario": scenario,
        "baseline_sha256": FROZEN_BASELINE_SHA256,
        "configuration_sha256": GD_CONFIG_SHA256,
        "selected_epoch": 5,
        "training_forget_rows": len(evidence["forget_predictions"]),
        "retained_test_rows": len(evidence["retained_predictions"]),
        **evidence["metrics"],
        **evidence["forgetting"],
        **evidence["runtime"],
    }
    required_outputs = [
        paths["adapter"] / "adapter_config.json",
        paths["head"],
        *[paths["result_dir"] / name for name in [
            "training_history.csv", "retained_test_metrics.csv",
            "retained_test_probabilities.csv", "forget_set_probabilities.csv",
            "truth_ratio_values.csv", "forgetting_metrics.csv",
            "runtime.json", "configuration.json",
        ]],
    ]
    missing_outputs = [path for path in required_outputs if not path.is_file()]
    if missing_outputs:
        raise RuntimeError(f"Refusing COMPLETE marker; missing outputs: {missing_outputs}")

    # COMPLETE is written last and atomically, after every evidence file exists.
    temporary_complete = paths["complete"].with_suffix(".json.tmp")
    temporary_complete.write_text(json.dumps(complete, indent=2), encoding="utf-8")
    temporary_complete.replace(paths["complete"])
    if paths["checkpoint"].is_file():
        paths["checkpoint"].unlink()
    return complete


In [ ]:
def evaluate_and_save(model, scenario, history, summary):
    evidence = evaluate_scenario_evidence(model, scenario, summary)
    return publish_scenario_result(
        model, scenario, history, summary, evidence
    )


In [ ]:
def run_gd_scenario(scenario):
    if scenario not in FINAL_SCENARIOS:
        raise ValueError(f"Not a prespecified final GD scenario: {scenario}")

    action = prepare_scenario(scenario)
    if action == "skip":
        print("Valid completed result retained without overwrite:", SCENARIO_LABELS[scenario])
        return json.loads(scenario_paths(scenario)["complete"].read_text(encoding="utf-8"))

    restore_trainable(MODEL, FROZEN_BASELINE_STATE)
    assert state_fingerprint(capture_trainable(MODEL)) == FROZEN_BASELINE_SHA256

    try:
        history, summary = train_gd(MODEL, scenario, resume=(action == "resume"))
        result = evaluate_and_save(MODEL, scenario, history, summary)
        print("COMPLETE:", SCENARIO_LABELS[scenario])
        return result
    finally:
        # This also runs after an error, preventing cross-scenario contamination.
        restore_trainable(MODEL, FROZEN_BASELINE_STATE)
        MODEL.zero_grad(set_to_none=True)
        clear_device_cache()
        assert state_fingerprint(capture_trainable(MODEL)) == FROZEN_BASELINE_SHA256


## 9. Final Gradient Difference Runs

Run these cells one at a time. They are intentionally left unexecuted in the submitted build.


### 9.1 Recipient Withdrawal

Expected training forget rows: **426**.


In [ ]:
recipient_result = run_gd_scenario(
    "recipient_withdrawal"
)


### 9.2 Invalid Consent

Expected training forget rows: **4,148**.


In [ ]:
invalid_result = run_gd_scenario(
    "invalid_consent"
)


### 9.3 Retention Expiry

Expected training forget rows: **6,262**.


In [ ]:
retention_result = run_gd_scenario(
    "retention_expiry"
)


## 10. Gradient Difference Results

The table below reads saved, completed results only. Display rounding does not change any stored values.


In [ ]:
saved_rows = []
for scenario in FINAL_SCENARIOS:
    paths = scenario_paths(scenario)
    if not paths["complete"].is_file():
        continue
    complete = json.loads(paths["complete"].read_text(encoding="utf-8"))
    if not valid_complete_payload(complete, scenario):
        continue
    metrics = pd.read_csv(paths["result_dir"] / "retained_test_metrics.csv").iloc[0]
    saved_rows.append({
        "Scenario": SCENARIO_LABELS[scenario],
        "Forget Rows": complete["training_forget_rows"],
        "Primary Epoch": complete["selected_epoch"],
        "PR-AUC": metrics["pr_auc"],
        "Balanced Accuracy": metrics["balanced_accuracy"],
        "BCE": metrics["binary_cross_entropy"],
        "F1": metrics["f1"],
        "AUROC": metrics["auroc"],
        "KS Statistic": complete["ks_statistic"],
        "KS p-value": complete["ks_p_value"],
        "Gradient Difference Training Time": complete["gradient_difference_training_seconds"],
        "Full Retraining Training Time": complete["full_retraining_training_seconds"],
        "Speed-up": complete["speed_up"],
    })

final_results = pd.DataFrame(saved_rows)
display(final_results.round(4))
print(f"Completed saved scenarios: {len(final_results)}/{len(FINAL_SCENARIOS)}")


## 11. Findings and Limitations

> **Findings will be derived from completed saved scenario results.**

Any interpretation must keep retained utility, forgetting evidence, computational efficiency, and deletion-request size separate. The next cell produces only a factual saved-results status; it does not manufacture conclusions before runs exist.


In [ ]:
if final_results.empty:
    display(Markdown(
        "### Retained Utility\nPending completed saved results.\n\n"
        "### Forgetting\nPending completed saved results.\n\n"
        "### Computational Efficiency\nPending completed saved results.\n\n"
        "### Effect of Deletion-Request Size\nPending completed saved results."
    ))
else:
    display(Markdown(
        f"### Retained Utility\nSaved evidence is available for {len(final_results)} scenario(s); "
        "interpret PR-AUC and the other utility metrics separately.\n\n"
        "### Forgetting\nInterpret each saved KS statistic against Full Retraining; do not claim exact erasure.\n\n"
        "### Computational Efficiency\nUse the hardware-qualified saved training times and speed-ups.\n\n"
        "### Effect of Deletion-Request Size\nCompare request size only after all three prespecified scenarios are complete."
    ))


### 11.1 Limitations

- Gradient Difference updates the task-specific LoRA and binary head, not Qwen's original pretrained base.
- Only one Qwen checkpoint and model family are evaluated.
- The kidney-transplant data are synthetic.
- Text serialisation changes the computational characteristics of the original tabular task.
- Runtime comparisons are hardware dependent.
- Full Retraining is a behavioural reference, not proof of physical data removal.
- Truth Ratio and KS provide behavioural evidence rather than proof of parameter-level erasure or legal compliance.
- The trajectory is a fixed five-epoch configuration, not a broad hyperparameter study.
